# 03 Tầng 2 — RAG Pipeline

**Mục tiêu:**
- Tạo **POI embeddings** từ mô tả ngữ nghĩa (tên, category, city, thông tin bổ sung)
- Xây **FAISS vector index** để tìm kiếm POI phù hợp nhanh
- Với mỗi user, tổng hợp **user profile vector** từ lịch sử ghé thăm
- Retrieve **top-k POI** phù hợp nhất → tạo **context** cho LLM (Tầng 3)
- Lưu context đã chuẩn bị vào file JSON

**Cải tiến so với bài báo:** Bài báo gốc chỉ dùng `int_u(c)` dựa trên category. RAG bổ sung semantic similarity — user thích POI theo ngữ nghĩa mô tả thực tế, không chỉ category label.

**Thư viện cần cài:**
```bash
pip install sentence-transformers faiss-cpu pandas numpy
```


## 1. Cài đặt thư viện

In [ ]:
# Chạy một lần để cài thư viện
import subprocess, sys

packages = ["sentence-transformers", "faiss-cpu", "numpy", "pandas"]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

print("✓ Đã cài xong các thư viện.")

## 2. Load dữ liệu từ Tầng 1

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Đường dẫn project ─────────────────────────────────────────────────────
current_path = Path.cwd()
project_root = current_path.parent if current_path.name.lower() == "notebooks" else current_path

raw_path    = project_root / "Datasets" / "Raw"
gf_path     = project_root / "Datasets" / "GraphFrames"   # output Tầng 1
rag_path    = project_root / "Datasets" / "RAG"
rag_path.mkdir(parents=True, exist_ok=True)

# ── Load từ Parquet (output Tầng 1) ───────────────────────────────────────
poi_df        = pd.read_parquet(gf_path / "vertices")
pagerank_df   = pd.read_parquet(gf_path / "pagerank")
user_int_df   = pd.read_parquet(gf_path / "user_interest")
dur_df        = pd.read_parquet(gf_path / "poi_duration")
profit_df     = pd.read_parquet(gf_path / "poi_profit")

# Load visit để biết lịch sử user
visit_list = []
for city_dir in sorted(raw_path.iterdir()):
    if not city_dir.is_dir():
        continue
    f = city_dir / "touristsVisits.csv"
    if f.exists():
        df = pd.read_csv(f)
        df["city"] = city_dir.name
        visit_list.append(df)
visit_pdf = pd.concat(visit_list, ignore_index=True)

print(f"POI        : {len(poi_df):,} rows")
print(f"PageRank   : {len(pagerank_df):,} rows")
print(f"UserInterest: {len(user_int_df):,} rows")
print(f"Visits     : {len(visit_pdf):,} rows")

## 3. Tạo POI text descriptions để tạo embedding

In [ ]:
# Kết hợp thông tin POI thành câu mô tả ngữ nghĩa
# Mẫu: "Camp Nou is a Sport point of interest in Barcelona at coordinates 41.38, 2.12."

poi_enriched = poi_df.merge(
    pagerank_df[["id", "pagerank_score"]],
    on="id", how="left"
).merge(
    dur_df[["city", "poiID", "dur_avg_min", "pop_count"]],
    on=["city", "poiID"], how="left"
)

poi_enriched["dur_avg_min"]  = poi_enriched["dur_avg_min"].fillna(30)
poi_enriched["pop_count"]    = poi_enriched["pop_count"].fillna(1)
poi_enriched["pagerank_score"] = poi_enriched["pagerank_score"].fillna(0)

def build_poi_description(row) -> str:
    name     = row["poiName"].replace("_", " ")
    category = row["category"]
    city     = row["city"]
    dur      = int(row["dur_avg_min"])
    pop      = int(row["pop_count"])
    lat      = round(row["lat"], 4)
    lon      = round(row["lon"], 4)
    return (
        f"{name} is a {category} attraction in {city}. "
        f"Visitors typically spend {dur} minutes here. "
        f"It has been visited {pop} times (location: {lat}, {lon})."
    )

poi_enriched["description"] = poi_enriched.apply(build_poi_description, axis=1)

print("Ví dụ mô tả POI:")
for desc in poi_enriched["description"].head(5):
    print(" •", desc)

## 4. Tạo embeddings bằng SentenceTransformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
import time

# all-MiniLM-L6-v2: nhỏ gọn, nhanh, phù hợp cho POI search
MODEL_NAME = "all-MiniLM-L6-v2"
print(f"Đang tải model {MODEL_NAME}...")
embedder = SentenceTransformer(MODEL_NAME)

descriptions = poi_enriched["description"].tolist()

print(f"Đang tạo embeddings cho {len(descriptions):,} POI...")
t0 = time.time()
poi_embeddings = embedder.encode(
    descriptions,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,  # normalize để dùng cosine similarity
)
print(f"✓ Embeddings shape: {poi_embeddings.shape} ({time.time()-t0:.1f}s)")

# Lưu embeddings
np.save(rag_path / "poi_embeddings.npy", poi_embeddings)
poi_enriched.to_parquet(rag_path / "poi_enriched.parquet", index=False)
print("✓ Đã lưu poi_embeddings.npy và poi_enriched.parquet")

## 5. Xây FAISS Index

In [ ]:
import faiss

embedding_dim = poi_embeddings.shape[1]  # 384 cho MiniLM

# IndexFlatIP: inner product (= cosine sim khi đã normalize)
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(poi_embeddings.astype(np.float32))

print(f"FAISS index: {faiss_index.ntotal:,} vectors, dim={embedding_dim}")

# Lưu index
faiss.write_index(faiss_index, str(rag_path / "poi_faiss.index"))
print("✓ Đã lưu poi_faiss.index")

## 6. Tạo User Profile Vector từ lịch sử ghé thăm

In [ ]:
def build_user_profile(
    user_id: str, city: str,
    visit_pdf, poi_enriched,
    user_int_df, embedder,
    budget_min: float = 480,
) -> dict:
    """
    Tổng hợp user profile:
    - visited_pois: danh sách POI đã ghé
    - top_categories: category ưa thích theo int_u(c)
    - profile_vector: embedding vector trung bình của các POI đã ghé
    - budget_min: ngân sách thời gian
    """
    # POI đã ghé của user trong city
    user_visits = visit_pdf[
        (visit_pdf["userID"] == user_id) & (visit_pdf["city"] == city)
    ]["poiID"].unique().tolist()

    # Category ưa thích
    user_cats = (
        user_int_df[
            (user_int_df["userID"] == user_id) & (user_int_df["city"] == city)
        ]
        .sort_values("int_u_c", ascending=False)
        .head(3)
    )
    top_cats = user_cats[["poiTheme", "int_u_c"]].to_dict("records")

    # Profile vector: trung bình embedding các POI đã ghé
    visited_descs = poi_enriched[
        (poi_enriched["city"] == city) &
        (poi_enriched["poiID"].isin(user_visits))
    ]["description"].tolist()

    if visited_descs:
        vecs = embedder.encode(visited_descs, normalize_embeddings=True)
        profile_vec = vecs.mean(axis=0)
        # Normalize lại sau mean
        norm = np.linalg.norm(profile_vec)
        if norm > 0:
            profile_vec = profile_vec / norm
    else:
        # Không có lịch sử: dùng embedding của category ưa thích
        cat_desc = f"A tourist interested in {', '.join([c['poiTheme'] for c in top_cats])} attractions in {city}."
        profile_vec = embedder.encode([cat_desc], normalize_embeddings=True)[0]

    # Tên các POI đã ghé
    visited_names = poi_enriched[
        (poi_enriched["city"] == city) &
        (poi_enriched["poiID"].isin(user_visits))
    ]["poiName"].apply(lambda x: x.replace("_", " ")).tolist()

    return {
        "user_id":       user_id,
        "city":          city,
        "visited_pois":  visited_names,
        "top_categories": top_cats,
        "profile_vector": profile_vec,
        "budget_min":    budget_min,
    }

print("Hàm build_user_profile đã sẵn sàng.")

## 7. Hàm Retrieve top-k POI

In [ ]:
def retrieve_top_k_pois(
    profile: dict,
    faiss_index,
    poi_enriched: pd.DataFrame,
    dur_df: pd.DataFrame,
    k: int = 10,
    exclude_visited: bool = True,
) -> pd.DataFrame:
    """
    Tìm top-k POI phù hợp nhất với user profile trong cùng city.
    Trả về DataFrame với cột: poiName, category, similarity, dur_avg_min, description
    """
    city = profile["city"]
    query_vec = profile["profile_vector"].astype(np.float32).reshape(1, -1)

    # Tìm k*3 để có đủ sau khi filter
    scores, indices = faiss_index.search(query_vec, k * 3)

    candidates = poi_enriched.iloc[indices[0]].copy()
    candidates["similarity"] = scores[0]

    # Chỉ lấy POI cùng city
    candidates = candidates[candidates["city"] == city]

    # Loại POI đã ghé
    if exclude_visited and profile["visited_pois"]:
        visited_set = set(v.replace(" ", "_") for v in profile["visited_pois"])
        candidates = candidates[~candidates["poiName"].isin(visited_set)]

    # Merge duration
    candidates = candidates.merge(
        dur_df[["city", "poiID", "dur_avg_min"]], on=["city", "poiID"], how="left"
    )
    candidates["dur_avg_min"] = candidates["dur_avg_min"].fillna(30)

    return candidates.head(k)[[
        "id", "poiName", "category", "similarity",
        "dur_avg_min", "lat", "lon", "description"
    ]].reset_index(drop=True)


print("Hàm retrieve_top_k_pois đã sẵn sàng.")

## 8. Chạy thử RAG cho một user

In [ ]:
DEMO_CITY = "Barcelona"

# Load lại FAISS index
faiss_index = faiss.read_index(str(rag_path / "poi_faiss.index"))
poi_enriched = pd.read_parquet(rag_path / "poi_enriched.parquet")

# Lấy user mẫu
demo_users = visit_pdf[
    visit_pdf["city"] == DEMO_CITY
]["userID"].value_counts().head(3).index.tolist()

for demo_user in demo_users:
    profile = build_user_profile(
        user_id    = demo_user,
        city       = DEMO_CITY,
        visit_pdf  = visit_pdf,
        poi_enriched = poi_enriched,
        user_int_df = user_int_df,
        embedder   = embedder,
        budget_min = 480,
    )

    retrieved = retrieve_top_k_pois(
        profile      = profile,
        faiss_index  = faiss_index,
        poi_enriched = poi_enriched,
        dur_df       = dur_df,
        k            = 8,
    )

    print(f"\n{'='*60}")
    print(f"User: {demo_user} | City: {DEMO_CITY}")
    print(f"Đã ghé: {profile['visited_pois']}")
    print(f"Top categories: {profile['top_categories']}")
    print(f"\nTop-8 POI đề xuất:")
    print(retrieved[["poiName", "category", "similarity", "dur_avg_min"]].to_string(index=False))

## 9. Tạo context JSON cho LLM (Tầng 3)

In [ ]:
import json

def build_llm_context(
    profile: dict,
    retrieved_pois: pd.DataFrame,
    graphframes_itinerary: list = None,  # kết quả BFS từ Tầng 1
) -> dict:
    """
    Tạo context hoàn chỉnh cho LLM:
    - user profile (lịch sử, sở thích, ngân sách)
    - top-k POI retrieved (ngữ nghĩa)
    - itinerary gợi ý từ GraphFrames (để LLM có tham chiếu)
    """
    context = {
        "user_id":   profile["user_id"],
        "city":      profile["city"],
        "budget_minutes": profile["budget_min"],
        "user_profile": {
            "visited_pois":   profile["visited_pois"],
            "top_categories": profile["top_categories"],
        },
        "retrieved_pois": [
            {
                "name":        row["poiName"].replace("_", " "),
                "category":    row["category"],
                "similarity":  round(float(row["similarity"]), 4),
                "avg_visit_min": round(float(row["dur_avg_min"]), 1),
                "description": row["description"],
            }
            for _, row in retrieved_pois.iterrows()
        ],
    }

    if graphframes_itinerary:
        context["graphframes_suggestion"] = [
            {
                "rank":       rank + 1,
                "path":       [p.replace("_", " ") for p in path],
                "profit":     round(profit, 4),
                "total_time_min": round(time_min, 1),
            }
            for rank, (path, profit, time_min) in enumerate(graphframes_itinerary)
        ]

    return context


print("Hàm build_llm_context đã sẵn sàng.")

In [ ]:
# Tạo context cho tất cả users mẫu và lưu file
CITIES_TO_EVAL  = ["Barcelona", "London", "Tokyo"]  # điều chỉnh theo dataset
USERS_PER_CITY  = 5   # số user mỗi city
TOP_K_RETRIEVE  = 10  # số POI retrieve
BUDGET_MIN      = 480 # 8 giờ

all_contexts = []

available_cities = poi_enriched["city"].unique().tolist()
eval_cities = [c for c in CITIES_TO_EVAL if c in available_cities]
if not eval_cities:
    eval_cities = available_cities[:3]  # fallback
    print(f"Dùng cities mặc định: {eval_cities}")

for city in eval_cities:
    city_users = (
        visit_pdf[visit_pdf["city"] == city]["userID"]
        .value_counts()
        .head(USERS_PER_CITY)
        .index.tolist()
    )

    print(f"\n── {city}: {len(city_users)} users ──")

    for user_id in city_users:
        profile = build_user_profile(
            user_id      = user_id,
            city         = city,
            visit_pdf    = visit_pdf,
            poi_enriched = poi_enriched,
            user_int_df  = user_int_df,
            embedder     = embedder,
            budget_min   = BUDGET_MIN,
        )

        retrieved = retrieve_top_k_pois(
            profile      = profile,
            faiss_index  = faiss_index,
            poi_enriched = poi_enriched,
            dur_df       = dur_df,
            k            = TOP_K_RETRIEVE,
        )

        context = build_llm_context(
            profile           = profile,
            retrieved_pois    = retrieved,
            graphframes_itinerary = None,  # thêm kết quả BFS nếu đã chạy Tầng 1
        )

        all_contexts.append(context)
        print(f"  ✓ {user_id}: {len(retrieved)} POI retrieved")

# Lưu tất cả context
context_file = rag_path / "llm_contexts.json"
with open(context_file, "w", encoding="utf-8") as f:
    json.dump(all_contexts, f, ensure_ascii=False, indent=2)

print(f"\n✓ Đã lưu {len(all_contexts)} contexts → {context_file}")

## 10. Kiểm tra nhanh context output

In [ ]:
# Preview context đầu tiên
ctx = all_contexts[0]
print(f"User    : {ctx['user_id']}")
print(f"City    : {ctx['city']}")
print(f"Budget  : {ctx['budget_minutes']} phút")
print(f"Đã ghé  : {ctx['user_profile']['visited_pois']}")
print(f"Top cats: {ctx['user_profile']['top_categories']}")
print(f"\nTop-10 POI retrieved:")
for i, poi in enumerate(ctx["retrieved_pois"][:10], 1):
    print(f"  {i:2d}. [{poi['category']:12s}] {poi['name']:35s} "
          f"sim={poi['similarity']:.3f}  ~{poi['avg_visit_min']:.0f}min")

print("\n✓ Tầng 2 (RAG Pipeline) hoàn thành.")
print(f"Output folder: {rag_path}")
for f in sorted(rag_path.iterdir()):
    size = f.stat().st_size
    print(f"  {f.name:35s} ({size/1024:.0f} KB)")